# **Assignment 3**


**Anja Pijak**  
ap72051@student.uni-lj.si


In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso, Ridge
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV


#### Data preparation


In [2]:
column_names = [
    "state", "county", "community", "communityname", "fold", "population", "householdsize",
    "racepctblack", "racePctWhite", "racePctAsian", "racePctHisp", "agePct12t21", "agePct12t29",
    "agePct16t24", "agePct65up", "numbUrban", "pctUrban", "medIncome", "pctWWage", "pctWFarmSelf",
    "pctWInvInc", "pctWSocSec", "pctWPubAsst", "pctWRetire", "medFamInc", "perCapInc", "whitePerCap",
    "blackPerCap", "indianPerCap", "AsianPerCap", "OtherPerCap", "HispPerCap", "NumUnderPov", 
    "PctPopUnderPov", "PctLess9thGrade", "PctNotHSGrad", "PctBSorMore", "PctUnemployed", 
    "PctEmploy", "PctEmplManu", "PctEmplProfServ", "PctOccupManu", "PctOccupMgmtProf", 
    "MalePctDivorce", "MalePctNevMarr", "FemalePctDiv", "TotalPctDiv", "PersPerFam", "PctFam2Par", 
    "PctKids2Par", "PctYoungKids2Par", "PctTeen2Par", "PctWorkMomYoungKids", "PctWorkMom", "NumIlleg", 
    "PctIlleg", "NumImmig", "PctImmigRecent", "PctImmigRec5", "PctImmigRec8", "PctImmigRec10", 
    "PctRecentImmig", "PctRecImmig5", "PctRecImmig8", "PctRecImmig10", "PctSpeakEnglOnly", 
    "PctNotSpeakEnglWell", "PctLargHouseFam", "PctLargHouseOccup", "PersPerOccupHous", 
    "PersPerOwnOccHous", "PersPerRentOccHous", "PctPersOwnOccup", "PctPersDenseHous", "PctHousLess3BR", 
    "MedNumBR", "HousVacant", "PctHousOccup", "PctHousOwnOcc", "PctVacantBoarded", "PctVacMore6Mos", 
    "MedYrHousBuilt", "PctHousNoPhone", "PctWOFullPlumb", "OwnOccLowQuart", "OwnOccMedVal", 
    "OwnOccHiQuart", "RentLowQ", "RentMedian", "RentHighQ", "MedRent", "MedRentPctHousInc", 
    "MedOwnCostPctInc", "MedOwnCostPctIncNoMtg", "NumInShelters", "NumStreet", "PctForeignBorn", 
    "PctBornSameState", "PctSameHouse85", "PctSameCity85", "PctSameState85", "LemasSwornFT", 
    "LemasSwFTPerPop", "LemasSwFTFieldOps", "LemasSwFTFieldPerPop", "LemasTotalReq", "LemasTotReqPerPop", 
    "PolicReqPerOffic", "PolicPerPop", "RacialMatchCommPol", "PctPolicWhite", "PctPolicBlack", 
    "PctPolicHisp", "PctPolicAsian", "PctPolicMinor", "OfficAssgnDrugUnits", "NumKindsDrugsSeiz", 
    "PolicAveOTWorked", "LandArea", "PopDens", "PctUsePubTrans", "PolicCars", "PolicOperBudg", 
    "LemasPctPolicOnPatr", "LemasGangUnitDeploy", "LemasPctOfficDrugUn", "PolicBudgPerPop", 
    "ViolentCrimesPerPop"
]


df = pd.read_csv("datasets/communities.data", names = column_names)

df.head()

# drop the first 5 columns (from assignment guidelines)
df = df.iloc[:, 5:]

# droping columns which are mostly na (from communities.names info)
# all of these have 1675 na rows out of (1994)
na_cols = ["LemasSwornFT", 
    "LemasSwFTPerPop", "LemasSwFTFieldOps", "LemasSwFTFieldPerPop", "LemasTotalReq", "LemasTotReqPerPop", 
    "PolicReqPerOffic", "PolicPerPop", "RacialMatchCommPol", "PctPolicWhite", "PctPolicBlack", 
    "PctPolicHisp", "PctPolicAsian", "PctPolicMinor", "OfficAssgnDrugUnits", "NumKindsDrugsSeiz", 
    "PolicAveOTWorked", "PolicCars", "PolicOperBudg", 
    "LemasPctPolicOnPatr", "LemasGangUnitDeploy", "PolicBudgPerPop"]
df = df.drop(columns=na_cols)

# replace ? with na
df.replace('?', np.nan, inplace=True)
# converting all columns in the df to numeric
df = df.apply(pd.to_numeric)
# fill na with average value
df.fillna(df.mean(), inplace=True)

df.head()



,population,householdsize,racepctblack,racePctWhite,racePctAsian,racePctHisp,agePct12t21,agePct12t29,agePct16t24,agePct65up,...,PctForeignBorn,PctBornSameState,PctSameHouse85,PctSameCity85,PctSameState85,LandArea,PopDens,PctUsePubTrans,LemasPctOfficDrugUn,ViolentCrimesPerPop
0,0.19,0.33,0.02,0.90,0.12,0.17,0.34,0.47,0.29,0.32,...,0.12,0.42,0.50,0.51,0.64,0.12,0.26,0.20,0.32,0.20
1,0.00,0.16,0.12,0.74,0.45,0.07,0.26,0.59,0.35,0.27,...,0.21,0.50,0.34,0.60,0.52,0.02,0.12,0.45,0.00,0.67
2,0.00,0.42,0.49,0.56,0.17,0.04,0.39,0.47,0.28,0.32,...,0.14,0.49,0.54,0.67,0.56,0.01,0.21,0.02,0.00,0.43
3,0.04,0.77,1.00,0.08,0.12,0.10,0.51,0.50,0.34,0.21,...,0.19,0.30,0.73,0.64,0.65,0.02,0.39,0.28,0.00,0.12
4,0.01,0.55,0.02,0.95,0.09,0.05,0.38,0.38,0.23,0.36,...,0.11,0.72,0.64,0.61,0.53,0.04,0.09,0.02,0.00,0.03


In [3]:
# train_data and test_data is calculated in previous code chunk
# here have to calculate x and y because we use built in lin. regression and not my methods
X = df.loc[:, df.columns != 'ViolentCrimesPerPop']
y = df['ViolentCrimesPerPop']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


# first test linear regression when all features are there, as a baseline
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
score = mean_squared_error(y_test, y_pred)
print("MSE before selecting features: ",score)


MSE before selecting features:  0.017528352717129242


In [4]:

# Define the parameter grid
param_grid = {'alpha': [0.001, 0.01, 0.1, 1, 10, 100, 1000]}

# Ridge Regression with Grid Search
ridge = Ridge()
ridge_cv = GridSearchCV(ridge, param_grid, cv=5, scoring='neg_mean_squared_error')
ridge_cv.fit(X_train, y_train)
ridge_best_alpha = ridge_cv.best_params_['alpha']
ridge_best_model = ridge_cv.best_estimator_

# Lasso Regression with Grid Search
lasso = Lasso(max_iter=10000)
lasso_cv = GridSearchCV(lasso, param_grid, cv=5, scoring='neg_mean_squared_error')
lasso_cv.fit(X_train, y_train)
lasso_best_alpha = lasso_cv.best_params_['alpha']
lasso_best_model = lasso_cv.best_estimator_

# Evaluate on the test set
ridge_test_mse = mean_squared_error(y_test, ridge_best_model.predict(X_test))
lasso_test_mse = mean_squared_error(y_test, lasso_best_model.predict(X_test))

print(f"Ridge Best Alpha: {ridge_best_alpha}, Test MSE: {ridge_test_mse}")
print(f"Lasso Best Alpha: {lasso_best_alpha}, Test MSE: {lasso_test_mse}")


Ridge Best Alpha: 1, Test MSE: 0.017525912823031873
Lasso Best Alpha: 0.001, Test MSE: 0.01698511409716176


In [5]:
# Get the coefficients from the Lasso model
lasso_coef = lasso_best_model.coef_
selected_features_lasso = X.columns[(lasso_coef != 0)].tolist()

print("Features selected by Lasso:", selected_features_lasso)

# Compare with forward attribute selection results from previous assignment
# (replace forward_selection_features with the list of features from your previous assignment)
forward_selection_features = ['PctKids2Par', 'racePctWhite', 'HousVacant', 'pctUrban', 'PctWorkMom', 'MalePctDivorce', 'PctIlleg', 'PctPersDenseHous', 'PctHousOccup', 'agePct12t29', 'LemasPctOfficDrugUn', 'MedOwnCostPctIncNoMtg', 'NumStreet', 'racepctblack', 'OtherPerCap', 'indianPerCap', 'pctWRetire', 'MalePctNevMarr', 'whitePerCap', 'PopDens', 'PctEmploy', 'pctWWage', 'PctPopUnderPov', 'PctEmplManu', 'perCapInc', 'RentLowQ', 'MedRent', 'PctVacMore6Mos', 'RentHighQ', 'LandArea', 'RentMedian', 'PctBSorMore', 'PctHousNoPhone', 'PctOccupManu', 'PctUsePubTrans']  # Replace with actual features
print("Features selected by Forward Selection:", forward_selection_features)

# Compare the lists
common_features = set(selected_features_lasso).intersection(forward_selection_features)
print("Common features selected by both methods:", common_features)




Features selected by Lasso: ['racepctblack', 'racePctWhite', 'pctUrban', 'pctWInvInc', 'AsianPerCap', 'PctEmplManu', 'MalePctDivorce', 'PctKids2Par', 'PctYoungKids2Par', 'PctWorkMom', 'PctIlleg', 'PctRecImmig10', 'PctPersDenseHous', 'HousVacant', 'PctHousOccup', 'PctVacantBoarded', 'MedRentPctHousInc', 'MedOwnCostPctIncNoMtg', 'NumStreet', 'PctForeignBorn', 'LemasPctOfficDrugUn']
Features selected by Forward Selection: ['PctKids2Par', 'racePctWhite', 'HousVacant', 'pctUrban', 'PctWorkMom', 'MalePctDivorce', 'PctIlleg', 'PctPersDenseHous', 'PctHousOccup', 'agePct12t29', 'LemasPctOfficDrugUn', 'MedOwnCostPctIncNoMtg', 'NumStreet', 'racepctblack', 'OtherPerCap', 'indianPerCap', 'pctWRetire', 'MalePctNevMarr', 'whitePerCap', 'PopDens', 'PctEmploy', 'pctWWage', 'PctPopUnderPov', 'PctEmplManu', 'perCapInc', 'RentLowQ', 'MedRent', 'PctVacMore6Mos', 'RentHighQ', 'LandArea', 'RentMedian', 'PctBSorMore', 'PctHousNoPhone', 'PctOccupManu', 'PctUsePubTrans']
Common features selected by both methods

In [6]:
model = LinearRegression()
# test linear regression with selected features
model.fit(X_train[selected_features_lasso], y_train)

y_pred1 = model.predict(X_test[selected_features_lasso])
score1 = mean_squared_error(y_test, y_pred1)
print("MSE after Lasso selection: ",score1)
# test linear regression with selected features
model.fit(X_train[forward_selection_features], y_train)

y_pred1 = model.predict(X_test[forward_selection_features])
score1 = mean_squared_error(y_test, y_pred1)
print("MSE after forward selection: ",score1)

MSE after Lasso selection:  0.017640642543213614
MSE after forward selection:  0.016469129226730366


In [7]:
# Load the dataset and select only white wine data
dfw = pd.read_csv('datasets/winequality-white.csv', delimiter=';')
dfw.fillna(dfw.mean(), inplace=True)
Xw = dfw.loc[:, dfw.columns != 'quality']
yw = dfw['quality']  # Target
dfw.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


In [8]:
from sklearn.discriminant_analysis import StandardScaler

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(Xw)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, yw, test_size=0.3, random_state=42)



In [15]:
X_train, X_test, y_train, y_test = train_test_split(Xw, yw, test_size=0.3, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mse_train = mean_squared_error(y_test, y_pred)


print(f"Train MSE: {mse_train}")

Train MSE: 0.5542927330136764


In [41]:
class RidgeRegressionGD:
    def __init__(self, learning_rate=0.00001, n_iterations=1000, alpha=0.01):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.alpha = alpha
    
    def fit(self, X, y):
        m, n = X.shape
        self.theta = np.zeros(n)
        self.cost_history = []

        for i in range(self.n_iterations):
            y_pred = X.dot(self.theta)
            error = y_pred - y
            gradient = (2/m) * X.T.dot(error) + 2 * self.alpha * self.theta
            self.theta -= self.learning_rate * gradient
            
            cost = (1/m) * np.sum(error**2) + self.alpha * np.sum(self.theta**2)
            self.cost_history.append(cost)
    
    def predict(self, X):
        return X.dot(self.theta)


# Standardize the data
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(Xw)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, yw, test_size=0.3, random_state=42)

# Train the model
ridge_gd = RidgeRegressionGD(learning_rate=0.000001, n_iterations=1000, alpha=0.01)
ridge_gd.fit(X_train, y_train)

# Predict and evaluate
y_pred = ridge_gd.predict(X_test)
print("Final cost:", mean_squared_error(y_test, y_pred))


Final cost: 35.54762094770268


In [42]:
class RidgeRegressionSGD:
    def __init__(self, learning_rate=0.01, n_iterations=1000, alpha=1.0):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.alpha = alpha
    
    def fit(self, X, y):
        m, n = X.shape
        self.theta = np.zeros(n)
        self.cost_history = []

        for i in range(self.n_iterations):
            cost = 0.0
            for j in range(m):
                index = np.random.randint(m)
                X_i = X[index, :].reshape(1, n)
                y_i = y[index]
                y_pred = X_i.dot(self.theta)
                error = y_pred - y_i
                gradient = (2 * X_i.T.dot(error)) + (2 * self.alpha * self.theta)
                self.theta -= self.learning_rate * gradient
                
                cost += (error**2) + self.alpha * np.sum(self.theta**2)
            self.cost_history.append(cost / m)
    
    def predict(self, X):
        return X.dot(self.theta)

# Train the model
ridge_sgd = RidgeRegressionSGD(learning_rate=0.01, n_iterations=1000, alpha=1.0)
ridge_sgd.fit(X_scaled, yw)

# Predict and evaluate
y_pred_sgd = ridge_sgd.predict(X_scaled)
print("Final cost (SGD):", ridge_sgd.cost_history[-1])


Final cost (SGD): [39.76779132]


In [45]:
def ridge_gradient_descent(X, y, alpha=0.01, learning_rate=0.0000001, num_iterations=1000):
    m, n = X.shape
    w = np.zeros(n)  # Initialize weights

    for _ in range(num_iterations):
        predictions = X.dot(w)
        errors = predictions - y
        
        # Compute the gradient
        gradient = (1 / m) * (X.T.dot(errors)) + alpha * w
        
        # Update parameters
        w -= learning_rate * gradient

    return w

In [10]:
def ridge_sgd(X, y, alpha=1.0, learning_rate=0.01, num_iterations=1000):
    m, n = X.shape
    theta = np.zeros(n)  # Initialize weights

    for iteration in range(num_iterations):
        # Shuffle the data at the start of each epoch
        indices = np.random.permutation(m)
        X_shuffled = X.iloc[indices].reset_index(drop=True)  # Reset index after shuffling
        y_shuffled = y.iloc[indices].reset_index(drop=True)
        
        # Perform SGD by going through each example
        for i in range(m):
            xi = X_shuffled.iloc[i].values.reshape(1, -1)  # Single example as 2D array
            yi = y_shuffled.iloc[i]

            # Prediction for the single example
            prediction = xi.dot(theta)
            error = prediction - yi

            # Compute the gradient for this example
            gradient = xi.flatten() * error + alpha * theta

            # Update the weights
            theta -= learning_rate * gradient


In [46]:
w = ridge_gradient_descent(X_train, y_train)
print(w)
# Make predictions on the train and test sets
y_train_pred = X_train.dot(w)
y_test_pred = X_test.dot(w)

# Calculate Mean Squared Error (MSE) for train and test sets
mse_train = mean_squared_error(y_train, y_train_pred)
mse_test = mean_squared_error(y_test, y_test_pred)

print(f"Train MSE: {mse_train}")
print(f"Test MSE: {mse_test}")

[-4.47496394e-06 -1.26084104e-05 -6.18229949e-06 -5.41581115e-06
 -2.05807452e-05  3.56739203e-07 -1.65377346e-05 -2.21658416e-05
  1.35490040e-05  2.90670942e-06  3.66142128e-05]
Train MSE: 35.24177261817765
Test MSE: 35.54891135672801
